In [ ]:
# STEP 2 — IMPORTS & BASE DIRECTORY SETUP

import os
from pathlib import Path
import pandas as pd

# Base directory containing all building folders
BASE_DIR = Path("/Users/jakegehrung/Desktop/data_raw/baseline_models")

# Optional: EPC dataset (not required for computation, but kept for reference)
EPC_PATH = Path("/Users/jakegehrung/Desktop/data_raw/data_processed/fintry_epc_hp_upgrade.csv")

# Load EPC (optional use / validation)
epc_df = pd.read_csv(EPC_PATH)

print(f"Loaded EPC dataset with {len(epc_df)} rows")
print(f"Scanning building folders in: {BASE_DIR}")

Loaded EPC dataset with 117 rows
Scanning building folders in: /Users/jakegehrung/Desktop/data_raw/baseline_models


In [ ]:
# STEP 3 — DEFINE ENERGY CALCULATION FUNCTION

def compute_energy_totals(df: pd.DataFrame) -> pd.DataFrame:
    """
    Takes an energy_results dataframe and computes total fuel columns.
    """

    # Fill NaNs with 0 for safe arithmetic
    df = df.fillna(0)

    # Electricity total
    df["electricity_tot_8"] = (
        df["electricity_heat_hp"]
        + df["electricity_hot_water_sol"]
        + df["electricity_app_light_vent"]
        - df["existing_pv"]
        - df["pv"]
    )

    # Gas and fuels
    df["lpg_tot_8"] = df["lpg_heat_hp"] + df["lpg_hot_water_sol"]
    df["mineral_wood_tot_8"] = df["mineral_wood_heat_hp"] + df["mineral_wood_hot_water_sol"]
    df["mains_gas_tot_8"] = df["mains_gas_heat_hp"] + df["mains_gas_hot_water_sol"]
    df["oil_tot_8"] = df["oil_heat_hp"] + df["oil_hot_water_sol"]

    # Biomass fuels
    df["wood_logs_tot_8"] = df["wood_logs_heat_hp"] + df["wood_logs_hot_water_sol"]
    df["wood_pellets_tot_8"] = df["wood_pellets_heat_hp"] + df["wood_pellets_hot_water_sol"]
    df["wood_chips_tot_8"] = df["wood_chips_heat_hp"] + df["wood_chips_hot_water_sol"]

    # Solid fuel
    df["coal_tot_8"] = df["coal_heat_hp"] + df["coal_hot_water_sol"]

    return df

In [3]:
# STEP 4 — LOOP THROUGH BUILDINGS AND UPDATE FILES

building_folders = [f for f in BASE_DIR.iterdir() if f.is_dir()]

print(f"Found {len(building_folders)} building folders")

updated_count = 0
missing_count = 0

for folder in building_folders:
    building_id = folder.name
    energy_file = folder / "TOTAL" / "energy_results.csv"

    if not energy_file.exists():
        print(f"[MISSING] {building_id}: energy_results.csv not found")
        missing_count += 1
        continue

    try:
        df = pd.read_csv(energy_file)

        df_updated = compute_energy_totals(df)

        df_updated.to_csv(energy_file, index=False)

        updated_count += 1

    except Exception as e:
        print(f"[ERROR] {building_id}: {e}")

print("\nDONE")
print(f"Updated files: {updated_count}")
print(f"Missing files: {missing_count}")

Found 117 building folders

DONE
Updated files: 117
Missing files: 0
